In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from functions import AlgPenDisc
from functions import AlgCCS
from functions import AlgRemez
from functions import AlgCS
from functions import AlgProxPen
from functions import vf
from functions import linear
from functions import quadratic
from functions import generic_fun
from functions import vf_concave
from functions import vf_maximum_concave

In [ ]:
def test(m,d,seed=0):
    np.random.seed(seed)
    res=np.zeros([4,5,d])
    k=0
    for d in np.arange(0,d):
        n=m
        c=np.random.rand(n)*10-5
        def fcp(x):
            return c.T@x
        def f(x):
            return c.T@x
        def df(x):
            return c
        A=np.zeros((2*n,n))
        A[:n,:n]=-np.eye(n)
        A[n:,:n]=np.eye(n)
        b=2*np.random.rand(2*n)
        X=[(linear,[A,2*np.ones(2*n)])]

        def norm2(y,param):
            j=param[0]
            return cp.norm(y[j],2)-1
            

        Y=[(generic_fun,[norm2,[i]]) for i  in range(2*n)]

        delta=np.random.rand()*0.2+0.1
        
        def gcp(x,y):
            return (A+delta*y)@x-b
        def g(x,y):
            return np.max((A+delta*y)@x-b)
        def dg(x,y):
            imax=np.argmax([(A+delta*y)[i]@x -b[i] for i in range(2*n) ])
            return (A+delta*y)[imax]
        def gcps(x,y):
            cs=[(A[i]+delta*y[i]).T@x-b[i] for i in range(2*n)]
            return cs
        Y0=[np.zeros((2*n,n))]
        x0=np.ones(n)
        
        rl=AlgProxPen(x0,Y0,fcp,gcp,g,X,Y,f=fcp,df=df,rho=1,concave=True,gcps=gcps,drop_cuts=True,gamma=1)
        res[0][1][k]=rl[2]
        res[1][1][k]=rl[3]
        res[2][1][k]=rl[4]
        res[3][1][k]=rl[5]

        
        rl=AlgMePa(x0,Y0,fcp,gcp,g,X,Y,concave=True,Ub=1000,gcps=gcps)
        res[0][2][k]=rl[2]
        res[1][2][k]=rl[3]
        res[2][2][k]=rl[4]
        res[3][2][k]=rl[4]
        
        rl=AlgBF(x0,Y0,fcp,gcp,g,X,Y,concave=True,gcps=gcps)
        res[0][3][k]=rl[2]
        res[1][3][k]=rl[3]
        res[2][3][k]=rl[4]
        res[3][3][k]=rl[4]
        
        rl=AlgRemez(x0,Y0,fcp,gcp,g,X,Y,concave=True,gcps=gcps)
        res[0][4][k]=rl[2]
        res[1][4][k]=rl[3]
        res[2][4][k]=rl[4]
        res[3][4][k]=rl[4]
        
        k+=1
    
    
   
    print("exp_lin_dim"+str(m)+"_seed"+str(seed))
    np.save("exp_lin_dim"+str(m)+"_seed"+str(seed), res)
    return res

def tables(res):
    print("times mean/median ProxLinDrop",np.mean(res[0][1]),np.median(res[0][1]))
    print("times mean/median CCS",np.mean(res[0][2]),np.median(res[0][2]))
    print("times mean/median CS",np.mean(res[0][3]),np.median(res[0][3]))
    print("times mean/median Remez",np.mean(res[0][4]),np.median(res[0][4]))

    print("feasibility mean/median ProxLinDrop",np.mean(res[1][1]),np.median(res[1][1]))
    print("feasibility mean/median CCS",np.mean(res[1][2]),np.median(res[1][2]))
    print("feasibility mean/median CS",np.mean(res[1][3]),np.median(res[1][3]))
    print("feasibility mean/median Remez",np.mean(res[1][4]),np.median(res[1][4]))

    print("iterations k mean/median ProxLinDrop",np.mean(res[3][1]),np.median(res[3][1]))
    print("iterations k mean/median CCS",np.mean(res[3][2]),np.median(res[3][2]))
    print("iterations k mean/median CS",np.mean(res[3][3]),np.median(res[3][3]))
    print("iterations k mean/median Remez",np.mean(res[3][4]),np.median(res[3][4]))

    print("calls vf mean/median ProxLinDrop",np.mean(res[2][1]),np.median(res[2][1]))
    print("calls vf mean/median CCS",np.mean(res[2][2]),np.median(res[2][2]))
    print("calls vf mean/median CS",np.mean(res[2][3]),np.median(res[2][3]))
    print("calls vf mean/median Remez",np.mean(res[2][4]),np.median(res[2][4]))

    plt.legend(["ProxLinDrop", "CCS","CS","Remez"])
    labels=["ProxLinDrop", "CCS","CS","Remez"]
    d=[res[0][1],res[0][2],res[0][3],res[0][4]]
    plt.boxplot(d,showfliers=False,showmeans=True, meanline=True, tick_labels=labels)
    

In [ ]:
#seed=np.random.randint(100000)
seed=60531
m=10
print(seed)
iters=100
res=test(m,iters,seed)

In [ ]:
res1=np.load("exp_lin_dim"+str(m)+"_seed"+str(seed)+".npy")
tables(res1)